<a href="https://colab.research.google.com/github/hicret11/paligemma2-resolution-experiment/blob/main/paligemma2_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PaliGemma 2: Does Higher Resolution Mean Better Eyes?

## Research Question
How does increasing image resolution from 224×224 to 448×448 affect
PaliGemma 2 3B on OCR, visual reasoning, and mixed read-and-reason tasks?

## Setup
- Model: PaliGemma 2 3B Mix
- Resolutions: 224×224 and 448×448\
- Images: 3
- Tasks per image: OCR, reasoning, mixed
- Total questions: 9 per model condition

In [ ]:
!pip -q install -U transformers accelerate huggingface_hub pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 54.8 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

login()

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [17]:
import os
print(os.listdir("/content"))

['.config', 'Image 01 – Menu.jpg', 'Image 3 Year-value chart.jpg', 'Image2 Turkish invoice.webp', 'sample_data']


In [18]:
from PIL import Image

menu_image = Image.open("/content/Image 01 – Menu.jpg").convert("RGB")
invoice_image = Image.open("/content/Image2 Turkish invoice.webp").convert("RGB")
chart_image = Image.open("/content/Image 3 Year-value chart.jpg").convert("RGB")

print("✅ All 3 images loaded!")

✅ All 3 images loaded!


In [19]:
import torch
from transformers import pipeline

MODEL_ID = "google/paligemma2-3b-mix-224"

pipe = pipeline(
    task="image-text-to-text",
    model=MODEL_ID,
    device=0,
    dtype=torch.bfloat16
)

print("✅ PaliGemma 2 3B–224 loaded!")

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

✅ PaliGemma 2 3B–224 loaded!


In [20]:
tests = [
    # IMAGE 1 — MENU
    {
        "image_name": "Menu",
        "image": menu_image,
        "task": "OCR",
        "question": "What is the price of Wagyu beef?",
        "expected": "$5.89"
    },
    {
        "image_name": "Menu",
        "image": menu_image,
        "task": "Reasoning",
        "question": "What is the cheapest main dish on the menu?",
        "expected": "Porterhouse ($5.69)"
    },
    {
        "image_name": "Menu",
        "image": menu_image,
        "task": "Mixed",
        "question": "How many main dishes cost less than $10?",
        "expected": "3"
    },

    # IMAGE 2 — TURKISH INVOICE
    {
        "image_name": "Turkish Invoice",
        "image": invoice_image,
        "task": "OCR",
        "question": "What is the invoice number?",
        "expected": "003"
    },
    {
        "image_name": "Turkish Invoice",
        "image": invoice_image,
        "task": "Reasoning",
        "question": "Which item has the highest total price?",
        "expected": "Ana Yemekler"
    },
    {
        "image_name": "Turkish Invoice",
        "image": invoice_image,
        "task": "Mixed",
        "question": "How many items were ordered in total?",
        "expected": "12"
    },

    # IMAGE 3 — CHART
    {
        "image_name": "Year-value Chart",
        "image": chart_image,
        "task": "OCR",
        "question": "What is the value shown for 2014?",
        "expected": "49"
    },
    {
        "image_name": "Year-value Chart",
        "image": chart_image,
        "task": "Reasoning",
        "question": "Which year has the highest value?",
        "expected": "2016"
    },
    {
        "image_name": "Year-value Chart",
        "image": chart_image,
        "task": "Mixed",
        "question": "How many years have a value greater than 30?",
        "expected": "4"
    }
]

for test in tests:
    result = pipe(
        test["image"],
        text="<image>" + test["question"],
        max_new_tokens=30
    )

    generated = result[0]["generated_text"]

    if test["question"] in generated:
        answer = generated.split(test["question"], 1)[-1].strip()
    else:
        answer = generated.strip()

    print("=" * 65)
    print("Image:", test["image_name"])
    print("Task:", test["task"])
    print("Question:", test["question"])
    print("Expected:", test["expected"])
    print("Answer:", answer)

[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Menu
Task: OCR
Question: What is the price of Wagyu beef?
Expected: $5.89
Answer: $11.99


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Menu
Task: Reasoning
Question: What is the cheapest main dish on the menu?
Expected: Porterhouse ($5.69)
Answer: baby beef ribs


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Image: Turkish Invoice
Task: Mixed
Question: How many items were ordered in total?
Expected: 12
Answer: 12


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Year-value Chart
Task: OCR
Question: What is the value shown for 2014?
Expected: 49
Answer: 60


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Year-value Chart
Task: Reasoning
Question: Which year has the highest value?
Expected: 2016
Answer: 2016
Image: Year-value Chart
Task: Mixed
Question: How many years have a value greater than 30?
Expected: 4
Answer: 4


In [21]:
import gc
import torch

if "pipe" in globals():
    del pipe

gc.collect()
torch.cuda.empty_cache()

print("✅ 224 model cleaned!")

✅ 224 model cleaned!


In [22]:
from transformers import pipeline
import torch

MODEL_ID = "google/paligemma2-3b-mix-448"

pipe = pipeline(
    task="image-text-to-text",
    model=MODEL_ID,
    device=0,
    dtype=torch.bfloat16
)

print("✅ PaliGemma 2 3B–448 loaded!")

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

✅ PaliGemma 2 3B–448 loaded!


In [23]:
for test in tests:
    result = pipe(
        test["image"],
        text="<image>" + test["question"],
        max_new_tokens=30
    )

    generated = result[0]["generated_text"]

    if test["question"] in generated:
        answer = generated.split(test["question"], 1)[-1].strip()
    else:
        answer = generated.strip()

    print("=" * 65)
    print("Image:", test["image_name"])
    print("Task:", test["task"])
    print("Question:", test["question"])
    print("Expected:", test["expected"])
    print("Answer:", answer)

[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Menu
Task: OCR
Question: What is the price of Wagyu beef?
Expected: $5.89
Answer: $5.89


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Menu
Task: Reasoning
Question: What is the cheapest main dish on the menu?
Expected: Porterhouse ($5.69)
Answer: new zealand lamb


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Image: Turkish Invoice
Task: Mixed
Question: How many items were ordered in total?
Expected: 12
Answer: 10


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Year-value Chart
Task: OCR
Question: What is the value shown for 2014?
Expected: 49
Answer: 49


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image: Year-value Chart
Task: Reasoning
Question: Which year has the highest value?
Expected: 2016
Answer: 2016
Image: Year-value Chart
Task: Mixed
Question: How many years have a value greater than 30?
Expected: 4
Answer: 4


## Results

The following table compares PaliGemma 2 3B at 224×224 and 448×448
using the same images, questions, and expected answers.

In [24]:
import pandas as pd

results = [
    ["Menu", "OCR", "$5.89", "$11.99", "$5.89", False, True],
    ["Menu", "Reasoning", "Porterhouse ($5.69)", "baby beef ribs", "new zealand lamb", False, False],
    ["Menu", "Mixed", "3", "3", "3", True, True],

    ["Turkish Invoice", "OCR", "003", "14", "003", False, True],
    ["Turkish Invoice", "Reasoning", "Ana Yemekler", "Main coure", "toplam", False, False],
    ["Turkish Invoice", "Mixed", "12", "12", "10", True, False],

    ["Year-value Chart", "OCR", "49", "60", "49", False, True],
    ["Year-value Chart", "Reasoning", "2016", "2016", "2016", True, True],
    ["Year-value Chart", "Mixed", "4", "4", "4", True, True],
]

df = pd.DataFrame(
    results,
    columns=[
        "Image",
        "Task",
        "Expected",
        "224 Answer",
        "448 Answer",
        "224 Correct",
        "448 Correct"
    ]
)

display(df)

df.to_csv("results.csv", index=False)

print("\n224 accuracy:", f"{df['224 Correct'].mean()*100:.1f}%")
print("448 accuracy:", f"{df['448 Correct'].mean()*100:.1f}%")
print("\n✅ results.csv saved!")

,Image,Task,Expected,224 Answer,448 Answer,224 Correct,448 Correct
0,Menu,OCR,$5.89,$11.99,$5.89,False,True
1,Menu,Reasoning,Porterhouse ($5.69),baby beef ribs,new zealand lamb,False,False
2,Menu,Mixed,3,3,3,True,True
3,Turkish Invoice,OCR,003,14,003,False,True
4,Turkish Invoice,Reasoning,Ana Yemekler,Main coure,toplam,False,False
5,Turkish Invoice,Mixed,12,12,10,True,False
6,Year-value Chart,OCR,49,60,49,False,True
7,Year-value Chart,Reasoning,2016,2016,2016,True,True
8,Year-value Chart,Mixed,4,4,4,True,True



224 accuracy: 44.4%
448 accuracy: 66.7%

✅ results.csv saved!


## Observations

- OCR improved from **0/3 correct at 224×224** to **3/3 correct at 448×448**.
- Overall strict accuracy increased from **4/9 (44.4%)** to **6/9 (66.7%)**.
- Higher resolution did not consistently improve reasoning and mixed tasks.
- On the Turkish invoice counting task, the 224 model answered correctly (12), while the 448 model answered incorrectly (10).
- These observations suggest that higher resolution helped the model read fine visual details in this small experiment, but did not guarantee better reasoning.

## Limitations

This was a small exploratory experiment with only three images and nine questions. It should not be treated as a benchmark or as evidence that higher resolution always improves PaliGemma 2. The experiment was designed to explore whether the resolution-related behavior discussed in the PaliGemma 2 paper could also be observed in a small Colab test.